# Trimorious 2.0 — Recovery to Hugging Face release
**M1.4.2 / M1.5 → M2.0 → M2.3 | staged, private by default | v1.0**

This notebook replaces the old teacher notebook for remaining work. It never runs teacher inference. Start with a fresh runtime and `STAGE='RECOVER'`; run all. Each stage is independently restartable. Change only the configuration cell, then Run all again. Use a T4 for SMOKE, TRAIN and EVALUATE. Do not run competing GPU notebooks. Save the notebook itself in Drive.

| Stage | Output / required action |
|---|---|
| RECOVER (CPU) | Verify exact saved corpus, recover 98 proposals for audit, independently recalculate numeric tasks; create review CSV and report. |
| APPROVE (CPU) | Fill review CSV (approve/reject each row, reviewer, notes), confirm licensing, approve exact reviewed data. Freeze release thresholds before training. |
| SMOKE (GPU) | Load pinned Qwen3 NF4, verify assistant labels, run 10 optimizer steps, discard test adapter. |
| TRAIN (GPU) | Fresh QLoRA adapter, resumable checkpoint, validation loss; save final adapter. |
| EVALUATE (GPU) | Reload adapter and base sequentially, identical frozen D07 scorer and prompts, plus separately supplied fresh objective confirmation set. |
| PACKAGE (CPU) | Prepare model card, license, hashes, inference example, private audit. Block unless approved evaluation thresholds and human review pass. |
| PUBLISH (CPU) | Explicit confirmation and HF_TOKEN Colab secret; create new private repo only, upload allowlisted files, verify remote hashes, then optionally make public. |

**Important:** Original archives, SQLite and v1 remain read-only. No automatic approval of teachers or narrative claims. Default training subset is deliberately small: 12 development and 6 validation per eligible template, only independently recomputed numerical math/science/statistics. Human review is still required. D04 tool traces are preserved but excluded from this SFT pilot (native tool format not validated). This is an experimental SFT scope, not a claim of comprehensive scientific intelligence. D07 stays private; a separately authored confirmation set and human rubric assessment are required for public release. No paid services or DPO/GRPO. GPU execution has not been tested by the notebook author.

In [ ]:
# @title 0. Choose one stage (start with RECOVER)
STAGE = 'RECOVER' # @param ['RECOVER','APPROVE','SMOKE','TRAIN','EVALUATE','PACKAGE','PUBLISH']
USE_DRIVE = True # @param {type:'boolean'}
INSTALL_GPU_DEPS = True # @param {type:'boolean'}
PROJECT_ROOT = '/content/drive/MyDrive/trimorious_PRIVATE_m1_4_1'
SOURCE_NAME = 'trimorious_m1_4_1_PRIVATE_GPU_partial_20261127T061605946898Z.zip'
D07_FILENAME = 'trimorious_m1_3_2_PRIVATE_full_1c3d70c27cc3c9f8.zip'
# Upload this older benchmark ZIP to /content or put it in PROJECT_ROOT. Required for EVALUATE.
FRESH_TEST_PATH = '' # @param {type:'string'}
# Fresh test JSONL format documented in section 5. Must be externally authored, sealed before evaluation; preferably before training.
FRESH_TEST_SHA256 = '' # @param {type:'string'}
FRESH_INDEPENDENCE_ATTESTED = False # @param {type:'boolean'}
# Confirm external authorship, expert key review, no access to this adapter's evaluation outputs, no tuning on the test.
DATA_LICENSE_REVIEWED = False # @param {type:'boolean'}
DATA_REVIEW_APPROVED = False # @param {type:'boolean'}
# By approving, you confirm rights to train and distribute the resulting adapter, and actual review of CSV rows.
HUMAN_EVAL_APPROVED = False # @param {type:'boolean'}
# After actual rubric review; see evaluation report. Not an automatic pass.
PUBLISH_APPROVAL = '' # @param {type:'string'}
MAKE_PUBLIC = False # @param {type:'boolean'}
PUBLIC_APPROVAL = '' # @param {type:'string'}
MAX_TRAIN_PER_TEMPLATE = 12
MAX_VAL_PER_TEMPLATE = 6
MAX_LENGTH = 512
GRAD_ACCUM = 16
EPOCHS = 1
SEED = 417031
# Provisional minimum release policy: approve before training; cannot change after dataset freeze.
MIN_D07_GAIN = 1.0       # percentage points, measurable diagnostic gain
MAX_FAMILY_DROP = 5.0   # percentage points
MIN_FRESH_GAIN = 0.0
MAX_PROTOCOL_DROP = 2.0
BASE = 'Qwen/Qwen3-4B-Instruct-2507'
REV = 'cdbee75f17c01a7cc42f958dc650907174af0554'
TARGET = 'roime81/trimorious-research-4B-v2'
EXPECTED_CORPUS = '9fe5e9dbce17096c4d1ddd28a82e45111a37cbba66c574ec63473f6f70dd283a'

In [ ]:
# @title 1. Mount Drive and common helpers (CPU)
import os, sys, json, hashlib, math, re, random, csv, zipfile, sqlite3, statistics, time, gc, shutil, platform
from pathlib import Path
from collections import Counter, defaultdict
from fractions import Fraction
from importlib import metadata
assert STAGE in {'RECOVER','APPROVE','SMOKE','TRAIN','EVALUATE','PACKAGE','PUBLISH'}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT=Path(PROJECT_ROOT); assert ROOT.is_dir(), f'Missing project folder: {ROOT}'
WORK=ROOT/'release_v2'; WORK.mkdir(exist_ok=True)
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False)
def sha(b): return hashlib.sha256(b if isinstance(b,bytes) else canon(b).encode()).hexdigest()
def put(p,obj):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_suffix(p.suffix+'.tmp'); tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False)); tmp.replace(p)
def get(p): return json.loads(Path(p).read_text())
def lines(b): return [json.loads(x) for x in b.splitlines() if x.strip()]
def readl(p): return lines(Path(p).read_text())
def writel(p,rows):
    p=Path(p); tmp=p.with_suffix(p.suffix+'.tmp'); tmp.write_text(''.join(canon(r)+'\n' for r in rows)); tmp.replace(p)
def writecsv(p,rows,fields):
    with Path(p).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
def archive_check(z):
    ns=z.namelist(); assert len(ns)==len(set(ns)) and z.testzip() is None
    m=json.loads(z.read('archive_manifest.json')); assert set(m)==set(ns)-{'archive_manifest.json'}
    assert all(sha(z.read(k))==v for k,v in m.items()), 'Archive hash mismatch'
    return m
def state(stage,**kw):
    p=WORK/'project_state.json'; s=get(p) if p.exists() else {}; s[stage]=kw; put(p,s)
def norm(s): return re.sub(r'\s+',' ',s.casefold()).strip()
def mask(s): return re.sub(r'(?<![a-z])[-+]?\d+(?:\.\d+)?(?:e[-+]?\d+)?','NUM',norm(s))
def close(a,b): return math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-10)
POLICY={'min_d07_gain_pp':MIN_D07_GAIN,'max_family_drop_pp':MAX_FAMILY_DROP,'min_fresh_gain_pp':MIN_FRESH_GAIN,'max_protocol_drop_pp':MAX_PROTOCOL_DROP}
print('Stage:',STAGE,'| Persistent work directory:',WORK)

## 2. Recover and verify (CPU)
No GPU or teacher calls. Independent recomputation uses only numbers in the authored prompt and standard formulas; this checks numerical values, not every derivation or unit. Reviewers must read the prompt, reference, units and assumptions. A fixed per-template cap limits superficial variants; not proof of semantic diversity. D07 lexical screening is inherited from the exact frozen corpus; no new examples or teacher text are added. Existing teacher proposals are preserved as private review evidence, not SFT. If the GPU ZIP is missing, stop rather than silently falling back to an older corpus.

In [ ]:
def recompute(r):
    # Only supported numerical templates; never execute prompt text as code.
    family=r['family']; t=int(r['template_id'].rsplit('_',1)[1]); q=r['prompt']
    nums=[float(x) for x in re.findall(r'(?<![\d.])[-+]?\d+(?:\.\d+)?',q)]
    if family=='mathematical_reasoning':
        if t==0: a,s=nums[-2:]; return (s+math.sqrt(s*s-4*a))/2
        if t==1: _,a,_,b,_,c=nums; return b+c+a/2
        if t==2: n,k=map(int,nums); return math.comb(n-2,k-2)
        if t==3: a,b,zero,c=nums; return max(abs(a),abs(c))
        if t==4:
            _,_,a,b,c=nums; b=abs(b); x=min(1,b/(2*a)); return a*x*x-b*x+c
        if t==5: a,b,_,_,c=nums; return abs(3*a+4*b-c)/5
        if t==6: a,b,c=map(int,nums); return pow(a,b,c)
        if t==7: a,b,c,_=nums; return a*a+2*a*b*c
        if t==8: raise ValueError('Use bounded matrix parser')  # dispatched below
        if t==9: a,_,b,c=nums; return math.fsum(a/b**i for i in range(int(c)))
    if family=='scientific_reasoning':
        if t==0: a,b,c=nums; return a*b*b/(2*c)
        if t==1: a,b,c=nums; return b*c/a
        if t==2: a,b,c=nums; return b*math.exp(-math.log(2)*c/a)
        if t==3: a,b=nums; return b*a*a/10000
        if t==4: a,b,c=nums; return b*(a-c)/a
        if t==5: a,b,c=nums; return a*(c*c-b*b)/2
        if t==6: a,b,c=nums; return a*c/(b+c)
        if t==7: a,b,c=nums; return a*1e-6/(b*c)
        if t==8: a,b,c=nums; return b*math.exp(a*c)
        if t==9: a,b,c=nums; return b*c/a
    if family=='statistical_reasoning':
        if t==0: n,k,_,z=nums; return (k+z*z/2)/(n+z*z)
        if t==1: n,a,m,b=nums; return math.sqrt(a*a/n+b*b/m)
        if t==2: n,k=nums; return 2*k*(n-k)/(n*(n-1))
        if t==3: return statistics.pvariance(nums)
        if t==4: return 1/nums[0]
        if t==5: n,p=nums; return n*p*(1-p)
        if t==6: a,b=nums; return a-b
        if t==7: a,b=nums[:5],nums[5:]; return statistics.mean(abs(x-y) for x,y in zip(a,b))
        if t==8: s=sum(nums); return -sum(x/s*math.log(x/s) for x in nums)
        if t==9: a,b,c,d=nums; return (a/c+b/d)/(1/c+1/d)
    raise ValueError('Unsupported template')
# Matrix prompts contain vertex numbers and nine entries; use an explicit bounded parser.
_recompute_basic=recompute
def recompute(r):
    if r['template_id']=='mathematical_reasoning_08':
        s=r['prompt'].split('matrix ',1)[1].split(', interpreting',1)[0]
        import ast
        a=ast.literal_eval(s); assert len(a)==3 and all(len(row)==3 for row in a)
        assert all(isinstance(v,int) and 0<=v<=100 for row in a for v in row)
        return sum(a[0][i]*a[i][2] for i in range(3))
    return _recompute_basic(r)

if STAGE=='RECOVER':
    source=ROOT/SOURCE_NAME; assert source.is_file(), f'Missing {source}'
    with zipfile.ZipFile(source) as z:
        manifest=archive_check(z); summary=json.loads(z.read('reports/summary.json'))
        all_candidates=[]
        for name in z.namelist():
            if (name.startswith('candidate_datasets/') and name.endswith('.jsonl')) or name=='quarantine/records.jsonl':
                all_candidates.extend(lines(z.read(name)))
        teacher_audit=lines(z.read('teacher_review/unapproved_proposals.jsonl'))+lines(z.read('quarantine/teacher_proposals.jsonl'))
    all_candidates.sort(key=lambda r:r['id'])
    assert len(all_candidates)==len({r['id'] for r in all_candidates})==5000
    excluded={'record_sha256','d07_screen','prior_pilot_char_similarity','disposition','diversity','split'}
    assert all(sha({k:v for k,v in r.items() if k not in excluded})==r['record_sha256'] for r in all_candidates)
    assert sha([r['record_sha256'] for r in all_candidates])==summary['corpus_sha256']==EXPECTED_CORPUS
    assert summary['d07']['questions_sha256']=='e724980d46b644f47952d3ab87789e94a32cb0b79835c4721ad3c7616c311c7e'
    assert all(r['training_approved'] is False for r in all_candidates)
    assert dict(Counter(r['split'] for r in all_candidates))=={'development_candidate':4000,'validation_candidate':1000}
    assert not any(r['d07_screen']['flagged'] for r in all_candidates)
    assert {r['template_id'] for r in all_candidates if r['split']=='development_candidate'}.isdisjoint({r['template_id'] for r in all_candidates if r['split']=='validation_candidate'})
    db=ROOT/'teacher_checkpoints.sqlite3'; db_counts={}
    if db.exists():
        with sqlite3.connect(db.resolve().as_uri()+'?mode=ro',uri=True) as conn:
            assert conn.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
            db_counts=dict(conn.execute("SELECT json_extract(payload,'$.teacher'),COUNT(*) FROM results GROUP BY 1"))
    eligible=defaultdict(list); ledger=[]; seen=set()
    for r in all_candidates:
        reason='excluded_narrative_or_tool'; check=None
        if r['family'] in {'mathematical_reasoning','scientific_reasoning','statistical_reasoning'} and r['split'] in {'development_candidate','validation_candidate'}:
            try:
                independent=recompute(r); m=re.search(r'Final result: ([^ ]+) \(',r['reference_response']); assert m
                check=bool(close(independent,r['numeric_answer']) and close(independent,r['independent_reference']) and close(independent,float(m[1])))
            except Exception as e: check=False
            if not check: reason='failed_independent_recheck'
            elif r['d07_screen']['flagged'] or r['verification']['passed'] is not True: reason='failed_prior_gate'
            elif norm(r['prompt']) in seen: reason='duplicate_prompt'
            else:
                reason='eligible_for_review'; seen.add(norm(r['prompt'])); eligible[(r['split'],r['template_id'])].append(r)
        ledger.append({'id':r['id'],'template':r['template_id'],'recomputed':check,'disposition':reason})
    chosen=[]
    for (split,template), group in sorted(eligible.items()):
        random.Random(SEED+sum(map(ord,template))).shuffle(group)
        cap=MAX_TRAIN_PER_TEMPLATE if split=='development_candidate' else MAX_VAL_PER_TEMPLATE
        chosen.extend(group[:cap])
    train_groups={r['template_id'] for r in chosen if r['split']=='development_candidate'}
    val_groups={r['template_id'] for r in chosen if r['split']=='validation_candidate'}
    assert train_groups and val_groups and train_groups.isdisjoint(val_groups)
    assert len(chosen)==len({r['id'] for r in chosen})
    chosen.sort(key=lambda r:r['id'])
    queue=[dict(id=r['id'],template=r['template_id'],split=r['split'],prompt=r['prompt'],response=r['reference_response'],unit=r['unit'],decision='',reviewer='',notes='') for r in chosen]
    fingerprint=sha(chosen)
    review_path=WORK/'review_queue.csv'
    if review_path.exists():
        assert (WORK/'recovery.json').exists() and get(WORK/'recovery.json')['review_set_sha256']==fingerprint, 'Existing review differs; preserve it and use a new workspace.'
        print('Existing review CSV preserved')
    else: writecsv(review_path,queue,list(queue[0]))
    writel(WORK/'review_candidates.jsonl',chosen); writel(WORK/'recovery_ledger.jsonl',ledger); writel(WORK/'teacher_audit_private.jsonl',teacher_audit)
    report={'source':source.name,'archive_sha256':sha(source.read_bytes()),'corpus_sha256':EXPECTED_CORPUS,'record_count':len(all_candidates),'manifest_files':len(manifest),'teacher_export_counts':dict(Counter(p['teacher'] for p in teacher_audit)),'teacher_database_counts':db_counts,'teacher_in_training':0,'recomputed_counts':dict(Counter(str(x['recomputed']) for x in ledger)),'dispositions':dict(Counter(x['disposition'] for x in ledger)),'review_counts':dict(Counter(r['split'] for r in chosen)),'review_set_sha256':fingerprint,'train_templates':sorted(train_groups),'validation_templates':sorted(val_groups),'training_approved':False}
    put(WORK/'recovery.json',report); state('RECOVER',complete=True,report='recovery.json'); print(json.dumps(report,indent=2)); print('NEXT: review',review_path,'then APPROVE. Do not approve unchecked rows.')

## 3. Dataset approval (CPU)
Open `release_v2/review_queue.csv` in a spreadsheet editor or download it, review each selected example, fill `decision` with `approve` or `reject`, `reviewer` with your identifier and `notes` with rationale. A blank decision excludes that record; rejected and blank records never train. Do not edit prompts or answers in the CSV; corrections require a new dataset revision and revalidation. Save the CSV back to Drive. Set the two approval flags only after actual review and rights confirmation. Preferably obtain a fresh externally authored sealed benchmark before training and record its SHA-256 without sharing it with dataset generation or checkpoint selection; set FRESH_TEST_SHA256. If unavailable, a separate researcher may create it after the final adapter is frozen, without seeing its evaluation outputs. Seal it before any fresh-test inference and never tune the adapter using it. Final publication is blocked until that independent evaluation passes. Default policy requires at least one percentage point diagnostic improvement, no fresh-test overall regression and bounded per-family regressions; stronger targets are desirable.

In [ ]:
def load_approved():
    approval=get(WORK/'dataset_approval.json'); assert approval['training_approved'] is True
    for name,digest in approval['files'].items(): assert sha((WORK/name).read_bytes())==digest, f'Changed approved file: {name}'
    assert approval['policy']==POLICY and approval['base']==BASE and approval['revision']==REV
    return approval

if STAGE=='APPROVE':
    assert DATA_LICENSE_REVIEWED and DATA_REVIEW_APPROVED, 'Review rights and examples first; then set both approval flags.'
    report=get(WORK/'recovery.json'); candidates=readl(WORK/'review_candidates.jsonl'); assert sha(candidates)==report['review_set_sha256']
    with (WORK/'review_queue.csv').open(newline='',encoding='utf-8-sig') as f: reviews=list(csv.DictReader(f))
    by_id={r['id']:r for r in reviews}; assert len(reviews)==len(by_id)==len(candidates) and set(by_id)=={r['id'] for r in candidates}
    approved={'train':[],'validation':[]}; dispositions=[]
    for r in candidates:
        q=by_id[r['id']]; assert q['prompt']==r['prompt'] and q['response']==r['reference_response'] and q['split']==r['split'] and q['template']==r['template_id']
        d=q['decision'].strip().lower(); assert d in {'','approve','reject'}
        if d: assert q['reviewer'].strip() and q['notes'].strip(), 'Each explicit decision requires reviewer and rationale.'
        dispositions.append({'id':r['id'],'decision':d or 'unreviewed','reviewer':q['reviewer'],'notes':q['notes']})
        if d=='approve':
            assert close(recompute(r),r['numeric_answer']) and not r['d07_screen']['flagged']
            # Use the source response as written, not unapproved teacher output.
            split='train' if r['split']=='development_candidate' else 'validation'
            approved[split].append({'id':r['id'],'template':r['template_id'],'family':r['family'],'messages':[{'role':'user','content':r['prompt']},{'role':'assistant','content':r['reference_response']}]})
    assert len(approved['train'])>=32 and len(approved['validation'])>=8, 'Review at least 32 development and 8 validation examples; these are smoke-level minima, not a quality guarantee.'
    assert {r['template'] for r in approved['train']}.isdisjoint({r['template'] for r in approved['validation']})
    assert not ({mask(r['messages'][0]['content']) for r in approved['train']} & {mask(r['messages'][0]['content']) for r in approved['validation']})
    if FRESH_TEST_SHA256: assert re.fullmatch('[a-f0-9]{64}',FRESH_TEST_SHA256)
    data_names=['train.jsonl','validation.jsonl','review_dispositions.jsonl']
    contents=[approved['train'],approved['validation'],dispositions]
    files={name:sha(''.join(canon(r)+'\n' for r in rows).encode()) for name,rows in zip(data_names,contents)}
    approval={'training_approved':True,'license_review_attested':True,'corpus_sha256':EXPECTED_CORPUS,'review_set_sha256':report['review_set_sha256'],'files':files,'counts':{k:len(v) for k,v in approved.items()},'policy':POLICY,'fresh_test_sha256':FRESH_TEST_SHA256,'base':BASE,'revision':REV,'scope':'human-reviewed, recomputed synthetic math/science/statistics; no teacher or tool trace SFT'}
    if (WORK/'dataset_approval.json').exists(): assert get(WORK/'dataset_approval.json')==approval, 'Approval already frozen. New revision requires separate workspace.'
    for name,rows in zip(data_names,contents): writel(WORK/name,rows)
    put(WORK/'dataset_approval.json',approval); state('APPROVE',complete=True,counts=approval['counts']); print(json.dumps(approval,indent=2)); print('NEXT: SMOKE')

## 4. GPU training — SMOKE then TRAIN
On a fresh T4. This uses Transformers Trainer with explicit assistant-only labels rather than relying on TRL version-specific behavior. Dependencies are installed only for GPU stages. If imports fail after installation, restart the session (not Drive), set the same stage and rerun. Never reduce MAX_LENGTH or change hyperparameters silently after approval/checkpoint creation. Overlong sequences fail closed, not truncated. Smoke uses a separate disposable adapter; full training always starts with a fresh adapter or resumes its own checkpoint. Model weights and optimizer checkpoints are stored in Drive. Default final checkpoint is selected before any D07 evaluation, not by D07. A tiny corpus may not improve the base model.

In [ ]:
if STAGE in {'SMOKE','TRAIN','EVALUATE'}:
    import subprocess
    if INSTALL_GPU_DEPS:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.6','peft>=0.17,<0.19','accelerate>=1.6,<2','bitsandbytes>=0.48,<1','huggingface_hub>=0.34,<1','safetensors>=0.5','tqdm','scipy','scikit-learn'])
    import torch
    from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig,Trainer,TrainingArguments,TrainerCallback,set_seed
    from transformers.trainer_utils import get_last_checkpoint
    from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel
    from tqdm.auto import tqdm
    assert torch.cuda.is_available(), 'Select a GPU runtime.'
    versions={x:metadata.version(x) for x in ['torch','transformers','peft','accelerate','bitsandbytes','huggingface_hub','safetensors']}
    print('GPU:',torch.cuda.get_device_name(0),'Versions:',versions)
    def load_base(train=False):
        gc.collect(); torch.cuda.empty_cache(); free,total=torch.cuda.mem_get_info()
        print(f'Free GPU: {free/2**30:.2f}/{total/2**30:.2f} GiB',flush=True)
        assert free/2**30>=10, 'Not enough free GPU memory. Use a clean runtime; do not load teacher models.'
        tok=AutoTokenizer.from_pretrained(BASE,revision=REV,trust_remote_code=False)
        assert tok.chat_template
        if tok.pad_token_id is None: tok.pad_token=tok.eos_token
        tok.padding_side='right'
        quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
        mdl=AutoModelForCausalLM.from_pretrained(BASE,revision=REV,quantization_config=quant,device_map={'':0},torch_dtype=torch.float16,attn_implementation='eager',low_cpu_mem_usage=True,trust_remote_code=False,use_safetensors=True)
        assert getattr(mdl,'is_loaded_in_4bit',False)
        mdl.config.use_cache=not train
        if train:
            mdl=prepare_model_for_kbit_training(mdl,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False})
            mdl=get_peft_model(mdl,LoraConfig(r=8,lora_alpha=16,lora_dropout=.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],revision=REV))
            mdl.print_trainable_parameters(); assert all('lora_' in name for name,p in mdl.named_parameters() if p.requires_grad)
        return mdl,tok
    class Rows(torch.utils.data.Dataset):
        def __init__(self,data): self.data=data
        def __len__(self): return len(self.data)
        def __getitem__(self,i): return self.data[i]
    def tokenize_rows(rows,tok):
        result=[]; too_long=[]
        for r in rows:
            msgs=r['messages']; prefix=tok.apply_chat_template(msgs[:-1],tokenize=True,add_generation_prompt=True); full=tok.apply_chat_template(msgs,tokenize=True,add_generation_prompt=False)
            assert full[:len(prefix)]==prefix, f'Chat template prefix mismatch: {r["id"]}'
            assert len(full)>len(prefix) and tok.eos_token_id in full[len(prefix):], f'No supervised EOS: {r["id"]}'
            if len(full)>MAX_LENGTH: too_long.append(r['id']); continue
            labels=[-100]*len(prefix)+full[len(prefix):]; assert all(x==-100 for x in labels[:len(prefix)]) and any(x!=-100 for x in labels[len(prefix):])
            result.append({'input_ids':full,'attention_mask':[1]*len(full),'labels':labels})
        assert not too_long, f'Would truncate {len(too_long)} answers; revise config before smoke: {too_long[:5]}'
        print('Tokenized',len(result),'max length',max(len(r['input_ids']) for r in result)); return Rows(result)
    def collate(items):
        n=max(len(r['input_ids']) for r in items); out={}
        for key,pad in [('input_ids',tokenizer.pad_token_id),('attention_mask',0),('labels',-100)]: out[key]=torch.tensor([r[key]+[pad]*(n-len(r[key])) for r in items],dtype=torch.long)
        return out
    class Health(TrainerCallback):
        def __init__(self): self.nonzero=False; self.logs=[]
        def on_pre_optimizer_step(self,args,state,control,model=None,**kw):
            grads=[p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
            assert grads and all(torch.isfinite(g).all().item() for g in grads), 'Missing or nonfinite adapter gradients'
            self.nonzero |= any(torch.count_nonzero(g).item()>0 for g in grads)
        def on_log(self,args,state,control,logs=None,**kw):
            if logs:
                for k in ('loss','eval_loss','train_loss','grad_norm'):
                    if k in logs: assert math.isfinite(float(logs[k])),f'Nonfinite {k}'
                row={**logs,'step':state.global_step,'peak_gpu_gib':torch.cuda.max_memory_allocated()/2**30}; self.logs.append(row); print(row,flush=True)
    def training_config(approval): return {'approval_sha256':sha(approval),'base':BASE,'revision':REV,'max_length':MAX_LENGTH,'grad_accum':GRAD_ACCUM,'epochs':EPOCHS,'seed':SEED,'lora_rank':8,'alpha':16,'lr':1e-4,'precision':'fp16_nf4','versions':versions}

In [ ]:
if STAGE in {'SMOKE','TRAIN'}:
    approval=load_approved(); cfg=training_config(approval); fingerprint=sha(cfg)[:20]
    run_dir=WORK/'sft_output'/fingerprint; run_dir.mkdir(parents=True,exist_ok=True)
    if (run_dir/'config.json').exists(): assert get(run_dir/'config.json')==cfg
    put(run_dir/'config.json',cfg)
    if STAGE=='TRAIN':
        smoke=get(run_dir/'smoke_pass.json'); assert smoke['passed'] is True and smoke['config']==cfg, 'Run matching SMOKE stage first.'
        assert not (run_dir/'training_complete.json').exists(), 'Training already completed; use EVALUATE.'
    set_seed(SEED); torch.cuda.reset_peak_memory_stats()
    model,tokenizer=load_base(train=True)
    tr=tokenize_rows(readl(WORK/'train.jsonl'),tokenizer); va=tokenize_rows(readl(WORK/'validation.jsonl'),tokenizer)
    example=collate([tr[0]]); assert (example['labels']==-100).any() and (example['labels']!=-100).any()
    health=Health(); smoke_mode=STAGE=='SMOKE'
    output=run_dir/('smoke_scratch' if smoke_mode else 'checkpoints'); output.mkdir(exist_ok=True)
    args=TrainingArguments(output_dir=str(output),per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=GRAD_ACCUM,gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},fp16=True,bf16=False,optim='paged_adamw_8bit',learning_rate=1e-4,max_grad_norm=1.0,warmup_ratio=.03,num_train_epochs=EPOCHS,max_steps=10 if smoke_mode else -1,logging_steps=1,logging_nan_inf_filter=False,eval_strategy='no' if smoke_mode else 'epoch',save_strategy='no' if smoke_mode else 'steps',save_steps=5,save_total_limit=2,save_safetensors=True,report_to='none',seed=SEED,data_seed=SEED,remove_unused_columns=False,prediction_loss_only=True,disable_tqdm=False)
    trainer=Trainer(model=model,args=args,train_dataset=tr,eval_dataset=va,data_collator=collate,callbacks=[health])
    resume=None if smoke_mode else get_last_checkpoint(str(output))
    if resume:
        p=Path(resume); assert all((p/f).is_file() for f in ['adapter_model.safetensors','adapter_config.json','trainer_state.json','optimizer.pt','scheduler.pt','rng_state.pth']), 'Latest checkpoint incomplete; inspect earlier checkpoint before resuming.'
    print('Resume:',resume,'| training rows:',len(tr),'| validation rows:',len(va),flush=True)
    result=trainer.train(resume_from_checkpoint=resume)
    assert math.isfinite(result.training_loss) and health.nonzero, 'No verified nonzero adapter gradients or invalid loss'
    if smoke_mode:
        assert trainer.state.global_step==10
        put(run_dir/'smoke_pass.json',{'passed':True,'config':cfg,'steps':10,'loss':result.training_loss,'logs':health.logs}); state('SMOKE',complete=True,run=str(run_dir)); print('SMOKE PASSED. Select TRAIN in a fresh runtime.')
    else:
        metrics=trainer.evaluate(); assert math.isfinite(metrics['eval_loss'])
        final=run_dir/'final_adapter'; final.mkdir(exist_ok=True); trainer.save_model(str(final)); tokenizer.save_pretrained(str(final))
        assert (final/'adapter_model.safetensors').is_file()
        config=get(final/'adapter_config.json'); assert config['base_model_name_or_path']==BASE and config.get('revision')==REV
        hashes={p.name:sha(p.read_bytes()) for p in final.iterdir() if p.is_file()}
        put(run_dir/'training_complete.json',{'config':cfg,'metrics':metrics,'steps':trainer.state.global_step,'train_loss':result.training_loss,'logs':health.logs,'adapter_hashes':hashes}); put(WORK/'active_run.json',{'run':run_dir.name}); state('TRAIN',complete=True,run=str(run_dir)); print('TRAINING COMPLETE. Select EVALUATE in a fresh runtime.')
    del trainer,model,tokenizer; gc.collect(); torch.cuda.empty_cache()

## 5. Evaluation (GPU; resumable)
Upload the previous M1.3.2 full ZIP to `/content` (or save in the project folder). Its original private questions, gold and scorer are verified. Run fresh base and adapter in the same quantization and prompt settings, one model at a time. The frozen 1,000 questions include 880 objective and 120 rubric items. D07 is a known diagnostic, not independent confirmation. No checkpoint selection on D07. Evaluation may take many GPU sessions; completed responses are saved every item.

**Fresh test:** independently authored JSONL, at least 100 objective questions, ≥10 distinct `template` groups, unique `id` and `question`, `family`, `kind` (`numeric` or `choice`), `gold`, `atol`, `rtol` (numeric). Use A/B/C/D choices within the question for choice items. Author/review outside this notebook; seal its hash before evaluation (preferably before training). Example schema only: `{"id":"fresh-001","template":"external_group_01","family":"mathematical_reasoning","kind":"numeric","question":"[independent question]","gold":0,"atol":1e-6,"rtol":1e-6}`. The example is not a test item. Leave path blank to run D07 alone; public release stays blocked. Do not use these test answers to revise training.

Evaluation measures strict protocol separately from operational correctness. For rubric tasks, review paired blinded answers using the actual task rubric and document accept/reject. The human review is not automated. A model can generate a valid-looking JSON response that is scientifically wrong.

In [ ]:
def active_run():
    approval=load_approved(); p=WORK/'sft_output'/get(WORK/'active_run.json')['run']; done=get(p/'training_complete.json')
    assert done['config']['approval_sha256']==sha(approval)
    final=p/'final_adapter'
    assert all(sha((final/k).read_bytes())==v for k,v in done['adapter_hashes'].items()), 'Adapter changed'
    return approval,p,done,final

if STAGE=='EVALUATE':
    approval,run_dir,training_done,final=active_run()
    assert not (approval['fresh_test_sha256'] or (run_dir/'fresh_test_seal.json').exists()) or FRESH_TEST_PATH, 'A sealed fresh test is configured. Provide its path to avoid repeating D07 evaluation.'
    d07=ROOT/D07_FILENAME
    if not d07.exists(): d07=Path('/content')/D07_FILENAME
    assert d07.is_file(), 'Upload original M1.3.2 full archive.'
    with zipfile.ZipFile(d07) as z:
        archive_check(z); pre='PRIVATE_DO_NOT_TRAIN/'; m=json.loads(z.read(pre+'manifest.json'))
        assert m['training_use'] is False and all(sha(z.read(pre+k))==v for k,v in m['file_hashes'].items())
        questions=lines(z.read(pre+'D07_questions.jsonl')); keys=lines(z.read(pre+'D07_gold_key.jsonl')); keymap={r['id']:r for r in keys}
        assert len(questions)==len(keys)==len(keymap)==1000
        assert all(all(keymap[q['id']].get(k,v)==v for k,v in q.items()) for q in questions)
        benchmark=[{**q,**keymap[q['id']]} for q in questions]
        assert sha(benchmark)==m['benchmark_sha256']=='47694f1b2f4dbd44e8f5b6e58962ad07af5c2a844e375249e74d785de0b51283'
        assert sha(z.read(pre+'D07_questions.jsonl'))=='e724980d46b644f47952d3ab87789e94a32cb0b79835c4721ad3c7616c311c7e'
    fresh=[]
    if FRESH_TEST_PATH:
        fresh_bytes=Path(FRESH_TEST_PATH).read_bytes(); fresh_digest=sha(fresh_bytes)
        assert FRESH_INDEPENDENCE_ATTESTED, 'Confirm independent authorship and answer-key review before evaluation.'
        expected=approval['fresh_test_sha256'] or FRESH_TEST_SHA256
        assert expected and fresh_digest==expected, 'Fresh test hash missing or mismatch.'
        assert not approval['fresh_test_sha256'] or not FRESH_TEST_SHA256 or FRESH_TEST_SHA256==expected
        seal={'sha256':fresh_digest,'adapter_hashes':training_done['adapter_hashes'],'independence_attested':True,'sealed_before_training':bool(approval['fresh_test_sha256'])}
        seal_path=run_dir/'fresh_test_seal.json'
        if seal_path.exists(): assert get(seal_path)==seal, 'Fresh confirmation already sealed for this frozen adapter.'
        fresh=lines(fresh_bytes); assert len(fresh)>=100 and len({r['template'] for r in fresh})>=10
        assert len({r['id'] for r in fresh})==len(fresh) and len({norm(r['question']) for r in fresh})==len(fresh)
        for r in fresh:
            assert r['kind'] in {'numeric','choice'} and all(isinstance(r[k],str) and r[k].strip() for k in ['id','question','template','family'])
            if r['kind']=='numeric': assert isinstance(r['gold'],(int,float)) and not isinstance(r['gold'],bool) and math.isfinite(r['gold']) and 0<=r['atol']<=.01 and 0<=r['rtol']<=.001
            else: assert r['gold'] in 'ABCD' and len(r['gold'])==1
        known=[r['messages'][0]['content'] for name in ['train.jsonl','validation.jsonl'] for r in readl(WORK/name)]+[r['question'] for r in benchmark]
        assert not ({mask(q) for q in known}&{mask(r['question']) for r in fresh}), 'Fresh masked question overlap'
        assert not ({r['id'] for r in fresh}&{r['id'] for r in benchmark})
        # Check lexical similarity without using gold. This is not semantic independence proof.
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity
        v=TfidfVectorizer(analyzer='char',ngram_range=(3,5)); x=v.fit_transform(known+[r['question'] for r in fresh]); sim=cosine_similarity(x[len(known):],x[:len(known)]).max()
        assert sim<.78,f'Fresh test near-overlap: {sim}'
        put(seal_path,seal)
    evaluation_config={'base':BASE,'revision':REV,'adapter_hashes':training_done['adapter_hashes'],'versions':versions,'benchmark_hash':sha(benchmark),'fresh_hash':fresh_digest if fresh else '', 'max_new_tokens':384,'seed':132,'attention':'eager','precision':'nf4_fp16','scorer':'v0.2'}
    eval_dir=run_dir/'evaluation'/sha(evaluation_config)[:20]; eval_dir.mkdir(parents=True,exist_ok=True); put(eval_dir/'config.json',evaluation_config)
    print('Evaluation rows:',len(benchmark),'+',len(fresh),'fresh; checkpoints:',eval_dir)

In [ ]:
if STAGE=='EVALUATE':
    def options(t): return dict(re.findall(r'^([ABCD])\. (.+)$',t['question'],re.M))

    def last_object(text):
        # Parse final JSON object (including nested tool args); never eval model output.
        text=str(text).strip()
        if len(text)>20000: return None
        def reject_nonfinite(s): raise ValueError('Non-finite JSON constant')
        dec=json.JSONDecoder(parse_constant=reject_nonfinite); candidates=[]
        for m in re.finditer(r'\{',text):
            try:
                obj,end=dec.raw_decode(text[m.start():]); candidates.append((m.start()+end,m.start(),obj))
            except (ValueError,RecursionError): pass
        if not candidates: return None
        _,_,obj=max(candidates,key=lambda x:(x[0],-x[1]))
        if not isinstance(obj,dict): return None
        try: canon(obj)
        except (ValueError,TypeError,OverflowError): return None
        return obj

    def numeric_ok(x,gold,atol,rtol):
        if isinstance(x,bool) or not isinstance(x,(int,float)): return False
        try: return math.isfinite(float(x)) and math.isclose(float(x),gold,abs_tol=atol,rel_tol=rtol)
        except (OverflowError,ValueError): return False

    def execute_tool(task, raw):
        obj=last_object(raw)
        if not obj or set(obj)!={'tool','args'} or obj['tool']!=task['tool'] or canon(obj['args'])!=canon(task['tool_args']):
            return {'ok':False,'error':'Invalid or incorrect tool call; no tool executed.'}
        args=obj['args']; name=obj['tool']
        # Exact argument matching limits execution to the small authored fixture for each task.
        if name=='mean': val=statistics.mean(args['values'])
        elif name=='sample_variance': val=statistics.variance(args['values'])
        elif name=='dot': val=sum(x*y for x,y in zip(args['a'],args['b']))
        else: return {'ok':False,'error':'Unknown tool'}
        return {'ok':True,'value':val}

    def score(task, raw, tool_result=None):
        obj=last_object(raw); protocol=bool(obj is not None and set(obj)=={'answer'})
        val=obj.get('answer') if protocol else None
        kind=task['kind']; correct=None
        if kind=='rubric': protocol=bool(protocol and isinstance(val,str) and val.strip())
        if kind=='choice': correct=bool(protocol and isinstance(val,str) and val.strip().upper()==task['gold'])
        elif kind in ('numeric','tool_numeric'):
            correct=bool(protocol and numeric_ok(val,task['gold'],task['atol'],task['rtol']))
            if kind=='tool_numeric': correct=bool(correct and tool_result and tool_result.get('ok'))
        return {'score':None if correct is None else int(correct),'protocol_ok':protocol,'parsed_answer':val,
                'review_status':'human_review_required' if kind=='rubric' else 'objective_scored'}

    def prompt_for(task):
        if task['kind']=='tool_numeric':
            return task['question']+'\nAvailable tools: mean(values: list of numbers), sample_variance(values: list of numbers), dot(a: list of numbers, b: list of numbers). First return only JSON {"tool":"tool_name","args":{...}} with the required arguments. You will receive a tool result next.'
        fmt={'numeric':'a JSON number (not a quoted string); give at least six significant digits if not exact', 'choice':'the single choice letter A, B, C or D', 'rubric':'your concise written response as a JSON string'}[task['kind']]
        return task['question']+'\nReturn only a JSON object with key "answer" containing '+fmt+'.'

    legacy_score=score
    NUM=r'[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?'
    def as_number(v):
        if isinstance(v,dict) and set(v)=={'value'}: v=v['value']
        if isinstance(v,bool): return None
        if isinstance(v,(float,int)):
            try: return float(v) if math.isfinite(float(v)) else None
            except OverflowError: return None
        if not isinstance(v,str) or len(v)>100: return None
        v=v.strip()
        try:
            if re.fullmatch(NUM,v): out=float(v)
            elif re.fullmatch(r'[+-]?\d+\s*/\s*[+-]?\d+',v):
                a,b=v.split('/'); out=float(Fraction(int(a),int(b)))
            else: return None
            return out if math.isfinite(out) else None
        except (ValueError,ZeroDivisionError,OverflowError): return None

    def parse_response(raw):
        raw=str(raw).strip()
        if len(raw)>20000: return None,None,False
        def no_dup(pairs):
            d={}
            for k,v in pairs:
                if k in d: raise ValueError('duplicate key')
                d[k]=v
            return d
        def reject(s): raise ValueError('nonfinite')
        wrapped=re.fullmatch(r'```(?:json)?\s*([\s\S]*?)\s*```',raw,re.I)
        text=wrapped.group(1).strip() if wrapped else raw
        # Recover only known presentation artifacts, not arbitrary prose.
        if not wrapped:
            text=re.sub(r'^```(?:json)?\s*','',text,flags=re.I).strip()
            text=re.sub(r'^\|\s*(?=\{)','',text).strip()
        try:
            obj=json.loads(text,object_pairs_hook=no_dup,parse_constant=reject)
            if isinstance(obj,dict): return obj.get('answer') if set(obj)=={'answer'} else None,obj,text==raw
            if isinstance(obj,(str,int,float)) and not isinstance(obj,bool): return obj,None,False
        except (ValueError,RecursionError):
            if re.fullmatch(r'[A-Da-d]|'+NUM+r'|[+-]?\d+\s*/\s*[+-]?\d+',text): return text,None,False
        return None,None,False

    def hardened(t,raw,tool_result=None):
        v,obj,whole=parse_response(raw); k=t['kind']; numeric=as_number(v)
        if k=='rubric':
            protocol=bool(whole and obj and isinstance(v,str) and v.strip())
            return dict(strict_protocol=protocol,answer_parseable=bool(isinstance(v,str) and v.strip()),content_correct=None,operational_correct=None,parsed_value=None)
        if k=='choice':
            match=None
            if isinstance(v,str):
                s=v.strip(); match=s.upper() if re.fullmatch('[A-Da-d]',s) else next((a for a,b in options(t).items() if s.casefold()==b.casefold()),None)
            protocol=bool(whole and obj and isinstance(v,str) and re.fullmatch('[A-D]',v))
            parseable=match is not None; correct=match==t['gold']; val=match
        else:
            protocol=bool(whole and obj and isinstance(v,(int,float)) and not isinstance(v,bool) and numeric is not None)
            parseable=numeric is not None; correct=bool(parseable and numeric_ok(numeric,t['gold'],t['atol'],t['rtol'])); val=numeric
        operational=correct and (k!='tool_numeric' or bool(tool_result and tool_result.get('ok')))
        return dict(strict_protocol=protocol,answer_parseable=parseable,content_correct=int(correct),operational_correct=int(operational),parsed_value=val)

    for t in benchmark:
        if t['kind']=='rubric': continue
        tool=execute_tool(t,canon({'tool':t['tool'],'args':t['tool_args']})) if t['kind']=='tool_numeric' else None
        assert hardened(t,canon({'answer':t['gold']}),tool)['operational_correct']==1
        assert hardened(t,'malformed',tool)['operational_correct']==0
        if t['kind']!='choice':
            assert hardened(t,canon({'answer':{'value':str(t['gold'])}}),tool)['content_correct']==1
            assert not hardened(t,canon({'answer':{'value':str(t['gold'])}}),tool)['strict_protocol']
            assert not hardened(t,'{"answer":true}',tool)['answer_parseable']
    assert as_number('1/4')==0.25 and as_number('1/0') is None
    assert parse_response('{"answer":1,"answer":2}')[0] is None
    print('Scoring self-tests passed for',sum(t['kind']!='rubric' for t in benchmark),'objective tasks')

In [ ]:
if STAGE=='EVALUATE':
    SYSTEM='You are being evaluated on scientific reasoning. Use only the supplied information. Follow the specified JSON output format. Do not invent evidence.'
    MAX_NEW_TOKENS=384
    def generate(mdl,tok,messages):
        prompt=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
        batch=tok(prompt,return_tensors='pt',add_special_tokens=False); batch={k:v.to(mdl.get_input_embeddings().weight.device) for k,v in batch.items()}
        torch.cuda.synchronize(); start=time.perf_counter()
        with torch.inference_mode(): out=mdl.generate(**batch,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,pad_token_id=tok.pad_token_id)
        torch.cuda.synchronize(); generated=out[0,batch['input_ids'].shape[1]:]
        return tok.decode(generated,skip_special_tokens=True),int(len(generated)),time.perf_counter()-start
    def evaluate_one(mdl,tok,t):
        messages=[{'role':'system','content':SYSTEM},{'role':'user','content':prompt_for(t)}]
        raw,n,elapsed=generate(mdl,tok,messages); first=result=None
        if t['kind']=='tool_numeric':
            first=raw; result=execute_tool(t,first)
            messages += [{'role':'assistant','content':first},{'role':'user','content':'Actual local tool execution result: '+canon(result)+'\nReturn only {"answer": numeric_value} using the observation. If execution failed, report null.'}]
            raw,n2,e2=generate(mdl,tok,messages); n+=n2; elapsed+=e2
        return {'id':t['id'],'family':t['family'],'template':t['template'],'kind':t['kind'],'response':raw,'tool_call_response':first,'tool_result':result,'generated_tokens':n,'elapsed_seconds':elapsed,**hardened(t,raw,result)}
    # One SQLite row per completed inference. No model text executed except original allowlisted tool fixture.
    db=eval_dir/'responses.sqlite3'
    with sqlite3.connect(db) as conn:
        conn.execute('CREATE TABLE IF NOT EXISTS responses (model TEXT, item TEXT, payload TEXT NOT NULL, PRIMARY KEY(model,item))'); conn.commit()
    def run_eval(label,tasks):
        with sqlite3.connect(db) as conn:
            completed={x[0] for x in conn.execute('SELECT item FROM responses WHERE model=?',(label,))}
        todo=[t for t in tasks if t['id'] not in completed]; print(label,'cached',len(completed),'pending',len(todo))
        if not todo: return
        mdl=tok=None
        try:
            set_seed(132); torch.cuda.reset_peak_memory_stats(); mdl,tok=load_base(train=False)
            if label=='trimorious_v2': mdl=PeftModel.from_pretrained(mdl,str(final),is_trainable=False)
            mdl.eval(); mdl.config.use_cache=True
            with sqlite3.connect(db) as conn:
                for t in tqdm(todo,desc=label):
                    row=evaluate_one(mdl,tok,t); row['peak_gpu_gib']=torch.cuda.max_memory_allocated()/2**30
                    conn.execute('INSERT INTO responses VALUES(?,?,?)',(label,t['id'],canon(row))); conn.commit()
        finally:
            del mdl,tok; gc.collect(); torch.cuda.empty_cache()
    tasks=benchmark+fresh
    for label in ['qwen3_base','trimorious_v2']: run_eval(label,tasks)
    with sqlite3.connect(db) as conn:
        results={label:{i:json.loads(row) for i,row in conn.execute('SELECT item,payload FROM responses WHERE model=?',(label,))} for label in ['qwen3_base','trimorious_v2']}
    assert all(set(v)=={t['id'] for t in tasks} for v in results.values())
    from scipy.stats import binomtest
    def comparison(tasks):
        obj=[t for t in tasks if t['kind']!='rubric']; a=[results['qwen3_base'][t['id']]['operational_correct'] for t in obj]; b=[results['trimorious_v2'][t['id']]['operational_correct'] for t in obj]
        n=len(a); wins=sum(x<y for x,y in zip(a,b)); losses=sum(x>y for x,y in zip(a,b)); groups=defaultdict(list)
        for t,x,y in zip(obj,a,b): groups[t['template']].append(y-x)
        rng=random.Random(132); clusters=list(groups.values()); draws=[]
        for _ in range(2000):
            sample=rng.choices(clusters,k=len(clusters)); draws.append(100*sum(map(sum,sample))/sum(map(len,sample)))
        draws.sort()
        return {'n':n,'base_pct':100*sum(a)/n,'v2_pct':100*sum(b)/n,'delta_pp':100*(sum(b)-sum(a))/n,'wins':wins,'losses':losses,'mcnemar_exact_p':float(binomtest(wins,wins+losses).pvalue) if wins+losses else 1.0,'template_bootstrap_95_ci':[draws[50],draws[1949]],'templates':len(clusters)}
    diag=comparison(benchmark); fam={f:comparison([t for t in benchmark if t['family']==f]) for f in sorted({t['family'] for t in benchmark if t['kind']!='rubric'})}; confirmation=comparison(fresh) if fresh else None
    protocol={label:100*sum(v[t['id']]['strict_protocol'] for t in benchmark)/len(benchmark) for label,v in results.items()}
    metrics={label:{'mean_latency_s':statistics.mean(v[t['id']]['elapsed_seconds'] for t in tasks),'mean_generated_tokens':statistics.mean(v[t['id']]['generated_tokens'] for t in tasks),'peak_gpu_gib':max(r['peak_gpu_gib'] for r in v.values())} for label,v in results.items()}
    checks={'d07_gain':diag['delta_pp']>=POLICY['min_d07_gain_pp'],'family_regression':all(x['delta_pp']>=-POLICY['max_family_drop_pp'] for x in fam.values()),'protocol_regression':protocol['trimorious_v2']-protocol['qwen3_base']>=-POLICY['max_protocol_drop_pp'],'fresh_confirmation':bool(confirmation and confirmation['delta_pp']>=POLICY['min_fresh_gain_pp'])}
    # Paired blinded rubric queue, stored privately. Mapping never published.
    queue=[]; mapping={}; rng=random.Random(132)
    for t in benchmark:
        if t['kind']!='rubric': continue
        labels=['qwen3_base','trimorious_v2']; rng.shuffle(labels); mapping[t['id']]=labels
        queue.append({'id':t['id'],'question':t['question'],'rubric':canon(t.get('rubric',t.get('gold'))),'answer_a':results[labels[0]][t['id']]['response'],'answer_b':results[labels[1]][t['id']]['response'],'a_acceptable':'','b_acceptable':'','reviewer':'','notes':''})
    qpath=eval_dir/'rubric_review_private.csv'
    if not qpath.exists(): writecsv(qpath,queue,list(queue[0]))
    put(eval_dir/'rubric_mapping_private.json',mapping)
    report={'evaluation_config':evaluation_config,'d07':diag,'families':fam,'fresh':confirmation,'protocol_pct':protocol,'runtime_metrics':metrics,'checks':checks,'objective_gate_passed':all(checks.values()),'rubric_pending':len(queue),'note':'D07 diagnostic, not independent; fresh set author and scientific validity require human attestation. Cluster intervals are descriptive, not proof.'}
    put(eval_dir/'evaluation_report.json',report); put(WORK/'active_evaluation.json',{'run':run_dir.name,'evaluation':eval_dir.name}); state('EVALUATE',complete=True,objective_gate=all(checks.values()),report=str(eval_dir/'evaluation_report.json')); print(json.dumps(report,indent=2)); print('NEXT: finish private rubric review before PACKAGE. If gate fails, do not publish.')

## 6. Package and publication (CPU)
Inspect the evaluation report; fill the private rubric review CSV with `yes` or `no` in both acceptability columns, reviewer and notes for every row. Human reviewer must verify scientific correctness and the independently authored fresh benchmark; set HUMAN_EVAL_APPROVED only after actual review. The code checks that rubric acceptance does not decline by more than the preapproved family tolerance. It cannot independently certify a reviewer's expertise.

Packaging creates an **adapter-only** release (not a merged standalone model). It includes the pinned base revision and an inference example (verify after publication); no private examples, D07, teacher outputs, optimizer state or secrets. Base licensing and original synthetic training rights must be reviewed; the Qwen3 base is Apache-2.0. Confirm publication separately. For public release set `PUBLISH_APPROVAL='PUBLISH TRIMORIOUS 2.0'`, create a fine-grained write token in Hugging Face and add it as `HF_TOKEN` in Colab Secrets. First upload is private; only set MAKE_PUBLIC and `PUBLIC_APPROVAL='MAKE TRIMORIOUS 2.0 PUBLIC'` after checking the exact files. The notebook refuses to overwrite an existing unrelated repo. No GitHub writes. If a publish interruption occurs before ownership marker is saved, inspect the new private repo manually instead of overwriting it.

In [ ]:
if STAGE in {'PACKAGE','PUBLISH'}:
    approval,run_dir,training_done,final=active_run()
    active=get(WORK/'active_evaluation.json'); assert active['run']==run_dir.name
    eval_dir=run_dir/'evaluation'/active['evaluation']; report=get(eval_dir/'evaluation_report.json')
    assert report['evaluation_config']['adapter_hashes']==training_done['adapter_hashes']
    assert report['objective_gate_passed'] and report['fresh'], 'Objective/independent confirmation gate failed. No publication.'
    seal=get(run_dir/'fresh_test_seal.json'); assert seal['sha256']==report['evaluation_config']['fresh_hash'] and seal['adapter_hashes']==training_done['adapter_hashes'] and seal['independence_attested']
    if approval['fresh_test_sha256']: assert seal['sha256']==approval['fresh_test_sha256']
    assert HUMAN_EVAL_APPROVED, 'Actual human scientific/rubric review and independent confirmation attestation required.'
    with (eval_dir/'rubric_review_private.csv').open(newline='',encoding='utf-8-sig') as f: rows=list(csv.DictReader(f))
    mapping=get(eval_dir/'rubric_mapping_private.json'); assert len(rows)==len(mapping)==120 and {r['id'] for r in rows}==set(mapping)
    # Ensure the reviewed text is the exact saved inference, not modified answers.
    with sqlite3.connect(eval_dir/'responses.sqlite3') as conn:
        reviewed_counts=Counter()
        for r in rows:
            assert r['reviewer'].strip() and r['notes'].strip()
            for col,label in zip(['a','b'],mapping[r['id']]):
                data=json.loads(conn.execute('SELECT payload FROM responses WHERE model=? AND item=?',(label,r['id'])).fetchone()[0]); assert r['answer_'+col]==data['response']
                ans=r[col+'_acceptable'].strip().lower(); assert ans in {'yes','no'}
                reviewed_counts[label]+=ans=='yes'
    rubric_pct={k:100*reviewed_counts[k]/120 for k in ['qwen3_base','trimorious_v2']}
    assert rubric_pct['trimorious_v2']-rubric_pct['qwen3_base']>=-POLICY['max_family_drop_pp'], 'Human rubric regression'
    assert DATA_LICENSE_REVIEWED, 'Reconfirm licensing and redistribution rights before packaging/publication.'
    package=run_dir/'release_package'; package.mkdir(exist_ok=True)
    if STAGE=='PACKAGE':
        allowed={'adapter_model.safetensors','adapter_config.json','tokenizer.json','tokenizer_config.json','special_tokens_map.json','added_tokens.json','vocab.json','merges.txt','chat_template.jinja'}
        for f in final.iterdir():
            if f.is_file() and f.name in allowed: shutil.copy2(f,package/f.name)
        assert (package/'adapter_model.safetensors').is_file() and (package/'adapter_config.json').is_file()
        # Fetch base license at the exact revision; fail closed if unavailable.
        import urllib.request
        with urllib.request.urlopen(f'https://huggingface.co/{BASE}/resolve/{REV}/LICENSE',timeout=60) as f: lic=f.read()
        assert b'Apache License' in lic and b'Version 2.0' in lic
        (package/'LICENSE').write_bytes(lic)
        public_report={'d07':report['d07'],'families':report['families'],'fresh':report['fresh'],'protocol_pct':report['protocol_pct'],'rubric_acceptance_pct':rubric_pct,'runtime_metrics':report['runtime_metrics'],'caveats':['D07 is a known synthetic diagnostic benchmark; not a blinded final test.','Fresh benchmark independence and review attested by user; sealed before training: '+str(seal['sealed_before_training']),'No native tool-call SFT, narrative teacher SFT or external literature corpus.','No claims of comprehensive scientific expertise.']}
        put(package/'benchmark_summary.json',public_report)
        put(package/'training_config.json',training_done['config']); (package/'requirements.txt').write_text('\n'.join(k+'=='+v for k,v in training_done['config']['versions'].items() if k!='torch')+'\n')
        card=f"""---
license: apache-2.0
base_model: {BASE}
base_model_relation: adapter
pipeline_tag: text-generation
library_name: peft
tags:
- scientific-research
- qlora
- experimental
---
# Trimorious 2.0 — experimental scientific LoRA adapter

Base: `{BASE}` at `{REV}`. This is a PEFT adapter, not standalone model weights. Load the exact base revision. Training uses 4-bit NF4 QLoRA, FP16 compute, assistant-only loss and a template-capped human-reviewed synthetic corpus ({approval['counts']['train']} train / {approval['counts']['validation']} validation). No teacher proposals or tool traces were used for SFT. No DPO or GRPO. The original Trimorious v1 is unchanged.

## Evaluation
D07 objective accuracy: base {report['d07']['base_pct']:.2f}% vs adapter {report['d07']['v2_pct']:.2f}% ({report['d07']['n']} tasks). Fresh confirmation: base {report['fresh']['base_pct']:.2f}% vs adapter {report['fresh']['v2_pct']:.2f}% ({report['fresh']['n']} tasks). Human rubric acceptance: base {rubric_pct['qwen3_base']:.2f}% vs adapter {rubric_pct['trimorious_v2']:.2f}%. See `benchmark_summary.json` for family scores, uncertainty, latency and limitations. No benchmark questions or answer keys are public.

## Usage
See `inference.py`. Requires a CUDA-compatible GPU and matching dependencies; use compatible PyTorch. Base model attribution: Qwen Team, Apache-2.0. The user has confirmed synthetic dataset rights and redistribution approval. Training data and private review artifacts are not distributed; full independent reproduction therefore requires authorized access.

## Limitations
Experimental, narrowly fine-tuned model; may hallucinate or regress on unseen problems. The known D07 benchmark is synthetic and not independent. Fresh confirmation and human review are limited in scope. Do not rely on outputs as sole scientific, clinical, engineering or safety-critical evidence. Verify calculations, sources and assumptions independently. Native tool calling is not newly trained or certified.
"""
        (package/'README.md').write_text(card); (package/'CHANGELOG.md').write_text('# Trimorious 2.0\n\n- New Qwen3-based experimental QLoRA adapter; original model retained.\n- Private diagnostic and independent confirmation evaluation completed.\n')
        inference=f"""import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
BASE={BASE!r}
REV={REV!r}
ADAPTER={TARGET!r}
tok=AutoTokenizer.from_pretrained(BASE,revision=REV)
q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
base=AutoModelForCausalLM.from_pretrained(BASE,revision=REV,quantization_config=q,device_map={{'':0}},torch_dtype=torch.float16)
model=PeftModel.from_pretrained(base,ADAPTER).eval()
prompt=tok.apply_chat_template([{{'role':'user','content':'Explain the difference between correlation and causation.'}}],tokenize=False,add_generation_prompt=True)
inputs=tok(prompt,return_tensors='pt',add_special_tokens=False).to(model.get_input_embeddings().weight.device)
with torch.inference_mode(): out=model.generate(**inputs,max_new_tokens=256,do_sample=False,pad_token_id=tok.eos_token_id)
print(tok.decode(out[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True))
"""
        (package/'inference.py').write_text(inference)
        public_names=allowed|{'LICENSE','benchmark_summary.json','training_config.json','requirements.txt','README.md','CHANGELOG.md','inference.py'}
        assert all(p.is_file() and p.name in public_names|{'artifact_manifest.json'} for p in package.iterdir())
        manifest={p.name:sha(p.read_bytes()) for p in package.iterdir() if p.name!='artifact_manifest.json'}; put(package/'artifact_manifest.json',manifest)
        put(run_dir/'package_approval.json',{'manifest_sha256':sha((package/'artifact_manifest.json').read_bytes()),'evaluation_sha256':sha(report),'rubric_review_sha256':sha((eval_dir/'rubric_review_private.csv').read_bytes()),'target':TARGET}); state('PACKAGE',complete=True,package=str(package)); print('RELEASE PACKAGE READY:',package,'| review files before PUBLISH')

In [ ]:
if STAGE=='PUBLISH':
    assert PUBLISH_APPROVAL=='PUBLISH TRIMORIOUS 2.0', 'Explicit upload approval required.'
    assert TARGET=='roime81/trimorious-research-4B-v2'
    approved=get(run_dir/'package_approval.json'); assert approved['target']==TARGET and approved['evaluation_sha256']==sha(report)
    assert approved['rubric_review_sha256']==sha((eval_dir/'rubric_review_private.csv').read_bytes())
    assert approved['manifest_sha256']==sha((package/'artifact_manifest.json').read_bytes())
    manifest=get(package/'artifact_manifest.json'); assert all(sha((package/k).read_bytes())==v for k,v in manifest.items())
    assert set(p.name for p in package.iterdir())==set(manifest)|{'artifact_manifest.json'}
    import subprocess
    subprocess.check_call([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.34,<1'])
    from huggingface_hub import HfApi,hf_hub_download
    from huggingface_hub.utils import RepositoryNotFoundError
    from google.colab import userdata
    token=userdata.get('HF_TOKEN'); assert token
    api=HfApi(token=token); assert api.whoami()['name']=='roime81', 'Wrong Hugging Face account.'
    receipt=run_dir/'publish_receipt.json'
    if not receipt.exists():
        api.create_repo(repo_id=TARGET,repo_type='model',private=True,exist_ok=False)
        put(receipt,{'target':TARGET,'manifest_sha256':approved['manifest_sha256'],'verified':False})
    rec=get(receipt); assert rec['target']==TARGET and rec['manifest_sha256']==approved['manifest_sha256']
    info=api.model_info(TARGET); assert info.private or rec['verified'], 'Unexpected public repository; stop.'
    names=set(manifest)|{'artifact_manifest.json'}
    remote=set(api.list_repo_files(TARGET,repo_type='model')); assert remote<=names|{'.gitattributes'}, 'Unexpected remote files; refusing to overwrite.'
    if not rec['verified']:
        api.upload_folder(repo_id=TARGET,repo_type='model',folder_path=str(package),allow_patterns=sorted(names),commit_message='Release Trimorious 2.0 experimental QLoRA adapter')
    remote_info=api.model_info(TARGET); commit=remote_info.sha; assert names<=set(api.list_repo_files(TARGET,revision=commit))
    for name in sorted(names):
        local=hf_hub_download(TARGET,name,revision=commit,token=token,force_download=True)
        assert sha(Path(local).read_bytes())==sha((package/name).read_bytes()),f'Remote hash mismatch: {name}'
    rec.update(verified=True,revision=commit,private=bool(remote_info.private)); put(receipt,rec)
    if MAKE_PUBLIC:
        assert PUBLIC_APPROVAL=='MAKE TRIMORIOUS 2.0 PUBLIC', 'Explicit public approval required.'
        api.update_repo_visibility(repo_id=TARGET,private=False,repo_type='model'); assert not api.model_info(TARGET).private
        rec['private']=False; put(receipt,rec)
    state('PUBLISH',complete=not rec['private'],private_uploaded_verified=True,url='https://huggingface.co/'+TARGET,revision=commit)
    print('Verified repository:', 'https://huggingface.co/'+TARGET,'| private:',rec['private'],'| revision:',commit)

## 7. Recovery checklist
- `release_v2/project_state.json` records stages, not assumptions. Keep the Drive folder private.
- If GPU OOM: restart clean runtime and rerun the same stage. Do not silently change a frozen training configuration. If you need a new sequence length or batch config, rerun SMOKE with the new config and start a new training run; never mix checkpoint configurations.
- If runtime interrupted during TRAIN: use same settings and dependencies. Last complete checkpoint is restored. If latest checkpoint is incomplete, preserve it and inspect previous checkpoint rather than guessing.
- If EVALUATE interrupted: rerun same stage with same dependency versions. SQLite resumes existing responses. Do not alter scoring.
- If dataset or evaluation gate fails, stop. Do not set flags merely to bypass the failure.
- To share results for review, share `recovery.json`, `dataset_approval.json`, `smoke_pass.json`, `training_complete.json`, and `evaluation_report.json` as appropriate. Never share private benchmark gold or HF_TOKEN.